In [4]:
import os
import json
import torch
from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration

input_dir = '/kaggle/input/datasets/yapeng111/enhanced-exdark-dataset/Enhanced_E_dataset'
output_json_path = '/kaggle/working/captions_E.json'

#加载 BLIP-2 模型
print("正在下载并加载 BLIP-2 模型..")

# 使用 float16 半精度加载模型
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b", 
    torch_dtype=torch.float16, 
    device_map="auto" # 自动分配到 GPU
)
print("✅ BLIP-2 模型加载完毕！")

#搜索图片
image_paths = []
for root, _, files in os.walk(input_dir):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_paths.append(os.path.join(root, file))

print(f"🔎 共找到 {len(image_paths)} 张图片准备生成描述。")

#批量推理
captions_dict = {} # 用于存储 {文件名: 文本描述}

if len(image_paths) > 0:
    print("开始生成图像文本描述...")
    
    # 我们也可以给模型一个提示词 (Prompt)，让它输出的格式更统一
    prompt = "A picture of" 
    
    for i, img_path in enumerate(image_paths):
        file_name = os.path.basename(img_path)
        
        try:
            image = Image.open(img_path).convert('RGB')
            
            # 防爆显存：缩放图片
            MAX_SIZE = 1200
            if max(image.size) > MAX_SIZE:
                resample_method = getattr(Image, 'Resampling', Image).LANCZOS
                image.thumbnail((MAX_SIZE, MAX_SIZE), resample_method)
            
            # 预处理并送入 GPU (保持 float16 半精度)
            inputs = processor(images=image, text=prompt, return_tensors="pt").to("cuda", torch.float16)
            
            # 生成文本 (max_new_tokens 控制句子长度)
            generated_ids = model.generate(**inputs, max_new_tokens=30)
            generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
            
            # 存入字典，例如: {"Bicycle_2015_00001.jpg": "a bicycle in a dark room next to a box"}
            # 把 prompt 拼回去，让句子完整
            captions_dict[file_name] = f"{prompt} {generated_text}"
            
        except Exception as e:
            print(f"❌ 处理图片 {file_name} 时出错: {e}")
            captions_dict[file_name] = "" # 出错则填空字符串
            
        # 每处理 100 张打印一次进度
        if (i + 1) % 100 == 0 or (i + 1) == len(image_paths):
            print(f"进度: 已处理 {i + 1} / {len(image_paths)} 张")

    # ================= 5. 保存为 JSON 文件 =================
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(captions_dict, f, indent=4)
        
    print(f"🎉 文本描述全部生成完毕！已保存至: {output_json_path}")
    
    # 抽查一下前 3 个结果
    print("\n🔍 抽查前 3 张图片的描述结果：")
    for k in list(captions_dict.keys())[:3]:
        print(f" - {k}: {captions_dict[k]}")

⏳ 正在下载并加载 BLIP-2 模型 (文件较大，大概需要 2-5 分钟，请耐心等待)...


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

进度: 已处理 7200 / 7363 张


The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.
The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.
The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when us

进度: 已处理 7300 / 7363 张


The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.
The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.
The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when us

进度: 已处理 7363 / 7363 张
🎉 文本描述全部生成完毕！已保存至: /kaggle/working/captions_E.json

🔍 抽查前 3 张图片的描述结果：
 - Cat_2015_03187.jpg: A picture of A picture of a cat in the dark
 - People_2015_06625.JPG: A picture of A picture of a street at night with cars parked on it
 - Boat_2015_00767.jpg: A picture of A picture of the city of amsterdam at night


In [5]:
import json

# 1. 读取刚才生成的带重复词的 JSON
json_path = '/kaggle/working/captions_E.json'
with open(json_path, 'r', encoding='utf-8') as f:
    captions = json.load(f)

# 2. 遍历字典，把重复的复读机短语替换掉
print("正在清理重复的提示词...")
for img_name, text in captions.items():
    # 替换各种大小写组合的重复
    clean_text = text.replace("A picture of A picture of", "A picture of")
    clean_text = clean_text.replace("A picture of a picture of", "A picture of")
    captions[img_name] = clean_text

# 3. 另存为一个干净的新 JSON 文件
clean_json_path = '/kaggle/working/captions_E_clean.json'
with open(clean_json_path, 'w', encoding='utf-8') as f:
    json.dump(captions, f, indent=4)

print(f"✅ 清理完成！已保存为: {clean_json_path}")

# 抽查清理后的效果
print("\n🔍 抽查清理后的前 3 条结果：")
for k in list(captions.keys())[:3]:
    print(f" - {k}: {captions[k]}")

正在清理重复的提示词...
✅ 清理完成！已保存为: /kaggle/working/captions_E_clean.json

🔍 抽查清理后的前 3 条结果：
 - Cat_2015_03187.jpg: A picture of a cat in the dark
 - People_2015_06625.JPG: A picture of a street at night with cars parked on it
 - Boat_2015_00767.jpg: A picture of the city of amsterdam at night
